# Prerequisites: Creating Sample Agents

## Overview

Let's first start by creating agents to be evaluated. This tutorial creates two sample agents for evaluation using different frameworks:
- [Strands Agents SDK](https://strandsagents.com/)
- [LangGraph](https://www.langchain.com/langgraph)

Both agents uses Anthropic Claude Haiku 4.5 from Amazon Bedrock as the LLM model but you can use any model of your preference and they have identical capabilities:
- **Math Tool**: Tool to perform basic math operations
- **Weather Tool**: Dummy implementation for weather tool


The architecture looks as following:

![Architecture](../images/agent_architecture.png)

## Prerequisites
- Python 3.10+
- AWS credentials

In [1]:
!pip install -r ../requirements.txt -q


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Deploy Agents

The `deploy_agents.py` script handles all infrastructure setup for both agents:
- Creates the AgentCore project scaffolds (if not already present)
- Copies the agent implementations into each project
- Deploys both agents to AgentCore Runtime
- Waits until both reach **READY** status
- Saves the agent IDs and ARNs to `agents_config.json`

Running the cell below will take **~5 minutes** on the first run. On subsequent runs the script detects the already-deployed agents and exits quickly.

In [2]:
!python deploy_agents.py

Region : us-east-1
Account: 849138760372

=== Strands Agent (acevalstrands2) ===


  Already deployed: acevalstrands2-xKJy20HJDc


  [acevalstrands2] status: READY

=== LangGraph Agent (acevallanggraph2) ===


  Already deployed: acevallanggraph2-xVzOqY40YB


  [acevallanggraph2] status: READY

Config saved to agents_config.json
  Strands   agent_id : acevalstrands2-xKJy20HJDc
  LangGraph agent_id : acevallanggraph2-xVzOqY40YB


## Load Agent Configuration

Read the agent IDs and ARNs written by `deploy_agents.py`.

In [3]:
import json
import uuid
import boto3

with open("agents_config.json") as f:
    _cfg = json.load(f)

region = _cfg["region"]
agent_id_strands = _cfg["strands"]["agent_id"]
agent_arn_strands = _cfg["strands"]["agent_arn"]
agent_id_langgraph = _cfg["langgraph"]["agent_id"]
agent_arn_langgraph = _cfg["langgraph"]["agent_arn"]

print(f"Region    : {region}")
print(f"Strands   : {agent_id_strands}")
print(f"LangGraph : {agent_id_langgraph}")

Region    : us-east-1
Strands   : acevalstrands2-xKJy20HJDc
LangGraph : acevallanggraph2-xVzOqY40YB


## Invoke the Strands Agent

Let's test the Strands agent by invoking the AgentCore Runtime endpoint using the `bedrock-agentcore` boto3 client.

When an agent session is created, AgentCore assigns it a unique `runtimeSessionId`. All subsequent turns in the same conversation pass the same session ID so the agent can maintain context across turns. The response is streamed back as a sequence of event chunks.

> **Learn more:** [Invoke an AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/invoke-agent-runtime.html)


In [4]:
session_id_strands = str(uuid.uuid4())
print(f"Strands session ID: {session_id_strands}")

Strands session ID: cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb


In [5]:
agentcore_dp = boto3.client("bedrock-agentcore", region_name=region)


def invoke_agent(agent_arn, prompt, session_id):
    """Invoke an AgentCore Runtime agent and return its text response."""
    response = agentcore_dp.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
    )
    raw = response["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: ") :]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw


result = invoke_agent(agent_arn_strands, "How much is 2+2?", session_id_strands)
print(result)

"2 + 2 = **4**"


In [6]:
result = invoke_agent(agent_arn_strands, "How is the weather now?", session_id_strands)
print(result)

"The weather right now is **sunny**! ☀️"


In [7]:
result = invoke_agent(
    agent_arn_strands, "Can you tell me the capital of the US?", session_id_strands
)
print(result)

"The capital of the United States is **Washington, D.C.** (Washington, District of Columbia).\n\nHowever, I should note that I don't have a specialized tool for answering general knowledge questions like this. I'm primarily equipped with a calculator for math and a weather tool. But I'm happy to help answer general questions based on my knowledge! 😊"


## Invoke the LangGraph Agent

The LangGraph agent was deployed alongside the Strands agent by `deploy_agents.py`. Let's test it.

Test the LangGraph agent with the same questions:

In [8]:
session_id_langgraph = str(uuid.uuid4())
print(f"LangGraph session ID: {session_id_langgraph}")

LangGraph session ID: 722e855f-5939-4aad-8afd-bc3d1a8a9ec6


In [9]:
result = invoke_agent(agent_arn_langgraph, "What is 2+2?", session_id_langgraph)
print(result)

"2 + 2 = **4**"


In [10]:
result = invoke_agent(
    agent_arn_langgraph, "What is the weather now?", session_id_langgraph
)
print(result)

"The weather right now is **sunny**! It's a nice day out. ☀️"


In [11]:
result = invoke_agent(
    agent_arn_langgraph, "Can you tell me the capital of the US?", session_id_langgraph
)
print(result)

"The capital of the United States is **Washington, D.C.** (District of Columbia). It's located on the east coast along the Potomac River and is home to the U.S. federal government, including the White House, the Capitol Building, and the Supreme Court."


In [12]:
print(f"Strands   agent_id={agent_id_strands}")
print(f"          agent_arn={agent_arn_strands}")
print(f"          session={session_id_strands}")
print(f"LangGraph agent_id={agent_id_langgraph}")
print(f"          agent_arn={agent_arn_langgraph}")
print(f"          session={session_id_langgraph}")

Strands   agent_id=acevalstrands2-xKJy20HJDc
          agent_arn=arn:aws:bedrock-agentcore:us-east-1:849138760372:runtime/acevalstrands2-xKJy20HJDc
          session=cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb
LangGraph agent_id=acevallanggraph2-xVzOqY40YB
          agent_arn=arn:aws:bedrock-agentcore:us-east-1:849138760372:runtime/acevallanggraph2-xVzOqY40YB
          session=722e855f-5939-4aad-8afd-bc3d1a8a9ec6


In [13]:
%store agent_id_strands
%store agent_arn_strands
%store session_id_strands
%store agent_id_langgraph
%store agent_arn_langgraph
%store session_id_langgraph

Stored 'agent_id_strands' (str)
Stored 'agent_arn_strands' (str)
Stored 'session_id_strands' (str)
Stored 'agent_id_langgraph' (str)
Stored 'agent_arn_langgraph' (str)
Stored 'session_id_langgraph' (str)


## Next Steps

Now that you have all the required pre-requisites, let's go through the individual evaluation tutorials:
Continue with the evaluation tutorials:
- [01-creating-custom-evaluators](../01-creating-custom-evaluators/): Create custom evaluators
- [02-running-evaluations](../02-running-evaluations/): Run on-demand and online evaluations
- [03-evaluation-workflows](../03-evaluation-workflows/): : Advanced techniques and dashboards